In [1]:
USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — PUR (18-Fold HUC-02) Physics-Informed

## Objective
Maximize the **out-of-sample Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** under the **18-Fold Leave-One-HUC02-Region-Out Prediction in Ungauged Regions (PUR)** cross-validation protocol on the evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (18-Fold HUC-02 PUR Cross-Validation)**:
  - `CV_FOLDS` (loaded from `pur_18fold_huc02_splits.json`) partitions the 531 basins by USGS 2-digit Hydrologic Unit Code (`HUC_02`, 18 regions).
  - In each fold `k`, an **entire HUC-02 region** (`fold['test_basin_mask']`) is held out as completely ungauged (`fold['qobs_fold_train']` is `NaN` on that region), and the model is trained on the remaining 17 HUC-02 regions (`fold['train_basin_mask']`) during `2003-10-01` to `2008-09-30`.
  - Predict out-of-sample discharge for `fold['test_idx']` and assemble the full `Q_PRED` array of shape `(531, 3653)` across all 18 regional folds.
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`).
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period on each fold's 17 training regions.
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Hydrological Physics & Architectural Guidance
You are encouraged to integrate physical hydrological principles and domain knowledge into the model architecture, feature engineering, and training objective:
1. **Conservation Principles & Mass-Conserving Architectures**: Consider mass-conserving recurrent structures (e.g., MC-LSTM) or explicit conceptual-neural storage cells that partition precipitation and snowmelt into tracked storage states and outgoing fluxes (evapotranspiration, quickflow, interflow, slow groundwater baseflow).
2. **Snow Accumulation, Melt & Soil Moisture Accounting**: Incorporate explicit degree-day / radiation-temperature snowpack accounting (tracking snow water equivalent SWE and liquid water delivery) and multi-rate exponential Antecedent Precipitation / Soil Moisture Indices (API).
3. **Catchment Conditioning (FiLM / Hypernetworks)**: Use static catchment attributes (aridity, snow fraction, soil conductivity, geology, slope, vegetation) to modulate hidden dynamics via Feature-wise Linear Modulation (FiLM) or regularized hypernetworks.
4. **Multi-Forcing Fusion & Input Uncertainty**: Leverage the 3 meteorological forcing products (`Daymet`, `Maurer`, `NLDAS`) and auxiliary SWE/daylength inputs by learning product-uncertainty weights and cross-forcing agreement features.

In [2]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "pur_18fold"
_splits_json = json.loads(_find_file("pur_18fold_huc02_splits.json").read_text(encoding="utf-8"))
CV_SPLITS = [v for _, v in sorted(_splits_json["folds"].items(), key=lambda kv: int(kv[1]["fold_index"]))]
assert len(CV_SPLITS) == 18, f"Expected 18 PUR folds, got {len(CV_SPLITS)}"

_basin_to_idx = {str(b): i for i, b in enumerate(BASINS)}
CV_FOLDS = []
_covered_test = np.zeros(len(BASINS), dtype=bool)
for _fold in CV_SPLITS:
    _fid = int(_fold["fold_index"])
    _test_idx = np.array([_basin_to_idx[str(b)] for b in _fold["test_basins"]], dtype=np.int64)
    _train_idx = np.array([_basin_to_idx[str(b)] for b in _fold["train_basins"]], dtype=np.int64)
    _test_mask = np.zeros(len(BASINS), dtype=bool)
    _test_mask[_test_idx] = True
    _train_mask = np.zeros(len(BASINS), dtype=bool)
    _train_mask[_train_idx] = True
    assert not np.any(_test_mask & _train_mask), f"Train/test overlap in PUR fold {_fid}"
    assert not np.any(_covered_test & _test_mask), f"Duplicate test basins in PUR fold {_fid}"
    _covered_test |= _test_mask
    CV_FOLDS.append({
        "fold_id": _fid,
        "held_out_huc02": str(_fold["huc_02"]),
        "train_basins": [str(b) for b in _fold["train_basins"]],
        "test_basins": [str(b) for b in _fold["test_basins"]],
        "train_idx": _train_idx,
        "test_idx": _test_idx,
        "train_basin_mask": _train_mask,
        "test_basin_mask": _test_mask,
        "qobs_fold_train": np.where(
            _train_mask[:, None] & ERA_TRAIN_MASK[None, :], QOBS_TRAIN, np.nan
        ).astype(np.float32),
    })
assert np.all(_covered_test), "18 PUR folds do not cover all 531 basins!"

print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}), {len(CV_FOLDS)} HUC-02 folds.")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (pur_18fold): 531 basins, 3653 days (1998-10-01 to 2008-09-30), 18 HUC-02 folds.
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [3]:
# Begin Stage

In [4]:
INSTRUCTION = [
    "In this 18-fold PUR benchmark (`CV_FOLDS`), an entire HUC-02 region is held out in each fold. Anchor cross-region extrapolation in universal physical hydrology: build a parameterless or weakly parameterized physical snowpack (SWE) and soil-moisture accounting module whose mass-balance dynamics remain valid in any unseen region.",
    "Exclude or strongly downweight raw spatial coordinates (`LAT`, `LON`) so the network cannot overfit to geographic location; instead, condition the model on physically transferable catchment functional properties (aridity index, snow fraction, soil conductivity, carbonate rock fraction, slope, elevation, LAI) via a regularized low-rank FiLM layer.",
    "Design a hybrid conceptual-neural architecture (inspired by dPL / delta-HBV / MC-LSTM) that explicitly tracks snow accumulation/melt, soil moisture degree-of-saturation, and multi-reservoir routing (quickflow, interflow, slow groundwater baseflow), ensuring physical plausibility when extrapolating to a new HUC-02 region.",
    "Engineer universal hydroclimatic regime features directly from the 365-day forcing history: dynamic Priestley-Taylor / radiation-temperature potential evapotranspiration proxy, rolling 30/90/365-day moisture surplus/deficit (`P - PET`), rain/snow partitioning, and multi-rate Antecedent Precipitation Indices.",
    "Regularize cross-regional training by penalizing variance in NSE across the 17 training HUC-02 regions (GroupDRO / regional variance penalty) alongside the basin-averaged NSE + log-discharge loss, forcing the model to learn relationships that generalize to the hardest unseen hydroclimatic regions.",
    "Combine multi-forcing fusion (dynamically weighting Daymet, Maurer, and NLDAS based on radiation/temperature/precipitation consistency) with exponential moving average (EMA) model weights to stabilize predictions in held-out regions.",
    "Use parallel multi-GPU execution across the 18 HUC-02 folds with mixed precision to train a diverse physics-informed ensemble (e.g., combining a FiLM-conditioned HydroLSTM and a conceptual-hybrid routing model) within the runtime budget.",
    "Synthesize the most effective physical extrapolation mechanisms (explicit snow & soil water balance, coordinate-free functional attribute conditioning, regional-variance-regularized loss, and multi-seed EMA ensembling across all 18 HUC-02 folds) to maximize PUR Median NSE."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [5]:
# [rewrite_cell]
# ==============================================================================
# Physics-Informed Scaled-MSE LSTM with Static Augmentation for PUR
# ==============================================================================

SEQ_LEN = 365
HIDDEN_SIZE = 128
DROPOUT = 0.4
STATIC_EMB_SIZE = 64
BATCH_SIZE = 512
LR = 1e-3
EPOCHS = 40
SAMPLES_PER_BASIN = 400

def engineer_features(dyn, aux, dates):
    """Physically-informed features including API and multi-forcing stats."""
    # Group forcings by variable (prcp, srad, tmax, tmin, vp)
    # prcp: 0, 5, 10 | srad: 1, 6, 11 | tmax: 2, 7, 12 | tmin: 3, 8, 13 | vp: 4, 9, 14
    p_all = dyn[..., [0, 5, 10]]
    p_mean = np.mean(p_all, axis=-1, keepdims=True)
    t_max = np.mean(dyn[..., [2, 7, 12]], axis=-1, keepdims=True)
    t_min = np.mean(dyn[..., [3, 8, 13]], axis=-1, keepdims=True)
    t_mean = (t_max + t_min) / 2.0
    srad = np.mean(dyn[..., [1, 6, 11]], axis=-1, keepdims=True)

    # Hargreaves PET (Physically based evaporation proxy)
    pet = 0.0023 * srad * (t_mean + 17.8) * np.sqrt(np.maximum(0, t_max - t_min))

    # Antecedent Precipitation Index (API) - Exponential decay mimics soil moisture
    def get_api(p, decays):
        B, T, _ = p.shape
        apis = []
        for d in decays:
            api = np.zeros((B, T, 1), dtype=np.float32)
            current = np.zeros((B, 1), dtype=np.float32)
            alpha = np.exp(-1.0 / d)
            for t in range(T):
                current = current * alpha + p[:, t, :]
                api[:, t, :] = current
            apis.append(api)
        return np.concatenate(apis, axis=-1)

    p_api = get_api(p_mean, [7, 30, 90, 180])

    # Seasonal Cycle
    doy = pd.to_datetime(dates).dayofyear.values
    sin_doy = np.sin(2 * np.pi * doy / 365.25)
    cos_doy = np.cos(2 * np.pi * doy / 365.25)
    season = np.tile(np.stack([sin_doy, cos_doy], axis=-1)[None, :, :], (dyn.shape[0], 1, 1))

    # Aux: Daylength and SWE
    dayl = np.mean(aux[..., [0, 2, 4]], axis=-1, keepdims=True)
    swe = np.mean(aux[..., [1, 3, 5]], axis=-1, keepdims=True)

    # Combine original forcing products with new features
    # This allows the LSTM to learn to weight different products
    feats = [dyn, p_mean, t_mean, pet, p_api, dayl, swe, season]
    return np.concatenate(feats, axis=-1).astype(np.float32)

class PURModel(nn.Module):
    def __init__(self, dyn_dim, stat_dim, hidden_size, dropout):
        super().__init__()
        self.static_encoder = nn.Sequential(
            nn.Linear(stat_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, STATIC_EMB_SIZE),
            nn.ReLU()
        )

        # FiLM-style parameter generation
        self.input_weight = nn.Linear(STATIC_EMB_SIZE, dyn_dim)
        self.input_bias = nn.Linear(STATIC_EMB_SIZE, dyn_dim)

        self.h0_gen = nn.Linear(STATIC_EMB_SIZE, hidden_size)
        self.c0_gen = nn.Linear(STATIC_EMB_SIZE, hidden_size)

        self.lstm = nn.LSTM(dyn_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_dyn, x_stat, augment=False):
        if augment and self.training:
            # Data Augmentation: Small noise on static features for PUR generalization
            x_stat = x_stat + torch.randn_like(x_stat) * 0.02

        emb = self.static_encoder(x_stat)

        # Modulate dynamic inputs
        gamma = self.input_weight(emb).unsqueeze(1)
        beta = self.input_bias(emb).unsqueeze(1)
        x_gated = x_dyn * (1 + gamma) + beta

        h0 = self.h0_gen(emb).unsqueeze(0).contiguous()
        c0 = self.c0_gen(emb).unsqueeze(0).contiguous()

        out, _ = self.lstm(x_gated, (h0, c0))
        return self.head(self.dropout(out)).squeeze(-1)

# Prepare Global Inputs
X_dyn_eng = engineer_features(DYNAMIC_INPUTS, AUX_FORCINGS, DATES)
X_stat_plus = np.concatenate([
    STATIC_ATTRIBUTES, np.log1p(BASIN_AREAS).reshape(-1, 1),
    LAT.reshape(-1, 1), LON.reshape(-1, 1)
], axis=-1)
X_stat_plus = np.nan_to_num(X_stat_plus, 0.0)
Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)

def train_fold(fold, device_id):
    dev = torch.device(f"cuda:{device_id}")
    tr_mask, te_idx = fold["train_basin_mask"], fold["test_idx"]

    # Local scaling for input features based on training basins
    d_m, d_s = X_dyn_eng[tr_mask].mean(axis=(0,1)), X_dyn_eng[tr_mask].std(axis=(0,1)) + 1e-6
    s_m, s_s = X_stat_plus[tr_mask].mean(axis=0), X_stat_plus[tr_mask].std(axis=0) + 1e-6
    Xd_norm = torch.from_numpy((X_dyn_eng - d_m) / d_s).to(dev)
    Xs_norm = torch.from_numpy((X_stat_plus - s_m) / s_s).to(dev)

    # Calculate basin-wise variance for NSE-style loss (Scaled MSE)
    Y_raw = fold["qobs_fold_train"]
    # We use std instead of variance to avoid extreme denominators
    basin_scales = np.nanstd(Y_raw, axis=1) + 1e-2
    # Normalize targets per-basin for more stable training gradient
    Y_target = torch.from_numpy(Y_raw).to(dev)
    scales_t = torch.from_numpy(basin_scales).to(dev)

    model = PURModel(Xd_norm.shape[-1], Xs_norm.shape[-1], HIDDEN_SIZE, DROPOUT).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, steps_per_epoch=1, epochs=EPOCHS
    )

    tr_indices = []
    for b in np.where(tr_mask)[0]:
        valid_t = np.where(ERA_TRAIN_MASK & ~np.isnan(Y_raw[b]))[0]
        valid_t = valid_t[valid_t >= SEQ_LEN - 1]
        if len(valid_t) > 0:
            sel = np.random.choice(valid_t, min(len(valid_t), SAMPLES_PER_BASIN), replace=False)
            tr_indices.extend([(b, t) for t in sel])

    for epoch in range(EPOCHS):
        model.train()
        random.shuffle(tr_indices)
        for i in range(0, len(tr_indices), BATCH_SIZE):
            batch = tr_indices[i:i+BATCH_SIZE]
            bb = torch.tensor([s[0] for s in batch], device=dev)
            tt = torch.tensor([s[1] for s in batch], device=dev)

            # Sequence indexing
            idx_range = tt.unsqueeze(1) + torch.arange(-SEQ_LEN + 1, 1, device=dev)
            y_hat_seq = model(Xd_norm[bb.unsqueeze(1), idx_range], Xs_norm[bb], augment=True)
            y_hat = y_hat_seq[:, -1]

            # Scaled MSE Loss: (y_hat - y_true)^2 / basin_std
            # This balances basins and aligns with NSE optimization
            sq_err = (y_hat - Y_target[bb, tt])**2
            loss = (sq_err / (scales_t[bb] + 1e-3)).mean()

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        preds_list = []
        for i in range(0, len(te_idx), 64):
            chunk_idx = te_idx[i:i+64]
            # Predict full sequence
            p_raw = model(Xd_norm[chunk_idx], Xs_norm[chunk_idx])
            preds_list.append(torch.clamp(p_raw, min=0).cpu().numpy())

    return te_idx, np.concatenate(preds_list, axis=0)

n_workers = max(1, NUM_GPUS)
with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = [executor.submit(train_fold, f, i % n_workers) for i, f in enumerate(CV_FOLDS)]
    for fut in futures:
        t_idx, p_val = fut.result()
        Q_PRED[t_idx] = p_val

/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [6]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (pur_18fold | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.67815 | Mean NSE: 0.22237 | IQR: [0.45906, 0.77468] | Basins < 0: 53
Candidate metric to be maximized: 0.678147


In [7]:
# End Stage